# 02: Feature Engineering

This notebook creates activity metrics and engineered features for HMM/HSMM modeling.

**IMPORTANT METHODOLOGICAL NOTES:**
1. Rolling features are computed **per individual** to avoid mixing data across subjects
2. Features are **standardized (z-scored)** before HMM to reduce noise and ensure comparable scales
3. Optional **log transformation** for activity to handle extreme values
4. Rolling features are for **modeling input**, NOT for global visualization across multi-year data

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from loguru import logger

from src import io, features

# Fix matplotlib overflow error with large datasets
plt.rcParams['agg.path.chunksize'] = 10000

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')

## 1. Load Cleaned Data

Load QC-validated data from previous step.

In [ ]:
df = io.load_raw_data(Path('../data/processed/qc_data.parquet'))
logger.info(f"Loaded data with shape: {df.shape}")
df.head()

## 2. Activity Metrics

In [ ]:
# Calculate activity metric from accelerometer data
# This uses Euclidean magnitude: sqrt(x^2 + y^2 + z^2)
if 'XAccel' in df.columns and 'YAccel' in df.columns and 'ZAccel' in df.columns:
    df['activity'] = features.calculate_accelerometer_activity(df)
    logger.success("Activity magnitude calculated from 3-axis accelerometer")
elif 'activity' not in df.columns:
    logger.warning("No accelerometer data or activity column found")

df.head()

## 3. Rolling Statistics

In [ ]:
# Add rolling statistics PER INDIVIDUAL
# This prevents rolling windows from spanning across different subjects
# NOTE: These features are for MODELING, not global visualization

if 'subject' in df.columns:
    df_features = features.add_rolling_statistics_per_individual(
        df,
        value_column='activity',
        subject_column='subject',
        windows=[6, 12, 24],
        stats=['mean', 'std', 'min', 'max']
    )
    logger.success("Rolling statistics computed per individual")
else:
    # Fallback to global rolling if no subject column
    df_features = features.add_rolling_statistics(
        df,
        value_column='activity',
        windows=[6, 12, 24],
        stats=['mean', 'std', 'min', 'max']
    )
    logger.warning("No subject column - using global rolling statistics")

logger.info(f"Features shape: {df_features.shape}")
df_features.head()

## 4. Time Features

In [ ]:
# Add time-based features
if 'timestamp' in df_features.columns:
    df_features = features.add_time_features(
        df_features,
        'timestamp',
        features=['hour', 'day_of_week', 'is_weekend']
    )

## 5. Lag Features

In [ ]:
# Add lag features
df_features = features.add_lag_features(
    df_features,
    'activity',
    lags=[1, 6, 24]
)

## 6. Log Transformation (Optional)

Log transformation reduces impact of extreme values and can improve HMM state identification.

In [ ]:
# Optional: log-transform activity to reduce impact of extreme values
df_features = features.log_transform_activity(df_features, 'activity', offset=1.0)
logger.info("Activity log-transformed")

# Show distribution comparison
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_features['activity'].dropna(), bins=50)
axes[0].set_title('Raw Activity Distribution')
axes[0].set_xlabel('Activity')
axes[1].hist(df_features['activity_log'].dropna(), bins=50)
axes[1].set_title('Log-transformed Activity Distribution')
axes[1].set_xlabel('Log(Activity + 1)')
plt.tight_layout()
plt.savefig('../outputs/activity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Feature Standardization

**CRITICAL for HMM:** Standardize features using z-scores to:
1. Reduce noise from raw sensor values
2. Ensure all features are on comparable scales
3. Improve HMM convergence and state identification

This addresses the "state flickering" problem.

In [ ]:
# Standardize features for HMM input
# Choose features to use for HMM (avoid using raw + log + rolling together)
hmm_features = ['activity_log']  # Use log-transformed activity

if 'subject' in df_features.columns:
    # Standardize per individual
    df_features = features.standardize_features(
        df_features,
        feature_columns=hmm_features,
        group_by='subject'
    )
    logger.success("Features standardized per individual")
else:
    # Global standardization
    df_features = features.standardize_features(
        df_features,
        feature_columns=hmm_features
    )
    logger.success("Features standardized globally")

# Show standardized features
print("\nStandardized feature statistics:")
for col in [f"{f}_standardized" for f in hmm_features]:
    if col in df_features.columns:
        print(f"{col}: mean={df_features[col].mean():.3f}, std={df_features[col].std():.3f}")

df_features.head()

## 8. Feature Summary

Summary of created features. Rolling features should NOT be visualized globally across multi-year data.

In [ ]:
# Feature summary
print(f"Total features created: {df_features.shape[1]}")
print(f"Total observations: {df_features.shape[0]}")
print(f"\nFeature columns:")
for col in df_features.columns:
    print(f"  - {col}")

# Check for NaN values
nan_counts = df_features.isna().sum()
print(f"\nNaN counts per column:")
print(nan_counts[nan_counts > 0])

## 9. Save Feature Data

In [ ]:
output_path = Path('../data/processed/feature_data.parquet')
io.save_processed_data(df_features, output_path, format='parquet')
logger.success(f"Feature data saved to {output_path}")

## 10. Log Parameters

In [ ]:
params = {
    'random_state': RANDOM_STATE,
    'n_samples': len(df_features),
    'n_features': len(df_features.columns),
    'rolling_windows': [6, 12, 24],
    'lag_periods': [1, 6, 24],
    'log_transformed': True,
    'standardized': True,
    'per_individual_rolling': 'subject' in df.columns,
}
logger.info(f"Feature engineering parameters: {params}")